<a href="https://colab.research.google.com/github/syedadilejazz/Complete_GenAI_Series_Bappy_euron/blob/main/Website_bot_using_Llama2%2CPinecone_%26_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Build Semantic Index means combining all the vectors together
#Semantic Search means qerying question embedding on vector DB and getting ranked result in return

#Install all the required libraries

In [2]:
!pip -q install -U langchain langchain-core langchain-community langchain-huggingface langchain_openai langchain_pinecone

!pip -q install -U bitsandbytes accelerate transformers

!pip -q install -U datasets loralib sentencepiece

!pip -q install -U pypdf

!pip -q install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.6 MB/s eta 0:00:00


In [3]:
# !pip install openai
# !pip insall tiktoken

In [4]:
#TO consider unstructured format in website
!pip -q install unstructured

In [5]:
!pip install tokenizers

In [6]:
!pip install xformers

In [7]:
!pip install pinecone-client

#Import all the Required Libraries

In [8]:
# from langchain.document_loaders import UnstructuredURLLoader
# from langchain.text_splitter import CharacterTextSplitter
# from langchain.embedding import OpenAIEmbeddings
# from lamgchain.chat_models import ChatOpenAI
# from langchain.vectorstored import Pinecone
# import pinecone
# from langchain.chains import RetrievalQAWithSourceChain
# from langchain.embeddings import HuggingFaceEmbeddings
# from transformers import AutoTokenizer, AutoModelForCausalLM
# from langchain.llms import HuggingFacePipeline
# from transformers import pipeline
# from huggingface_hub import notebook_login
# import textwrap
# import sys
# import os
# import torch

In [9]:
# Web/document loading
from langchain_community.document_loaders import UnstructuredURLLoader

# Text splitting
from langchain_text_splitters import CharacterTextSplitter

# OpenAI
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# Pinecone
from langchain_pinecone import PineconeVectorStore
import pinecone

# Hugging Face
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline

# Transformers
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Hugging Face Hub
from huggingface_hub import notebook_login

# Utilities
import textwrap
import sys
import os
import torch

/tmp/ipykernel_5355/1214564660.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredURLLoader


In [10]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

#Pass the URLS and extract the data from these URLs

In [18]:
#Single or Multiple URLS can be passed here
URLs=[
    'https://blog.gopenai.com/paper-review-llama-2-open-foundation-and-fine-tuned-chat-models-23e539522acb',
    'https://www.mosaicml.com/blog/mpt-7b',
    'https://stability.ai/blog/stability-ai-launches-the-first-of-its-stablelm-suite-of-language-models',
    'https://lmsys.org/blog/2023-03-30-vicuna/'
    ]

In [14]:
loaders=UnstructuredURLLoader(urls=URLs)
data=loaders.load()

In [15]:
type(data)

list

In [17]:
len(data)

4

In [16]:
data[0]

Document(metadata={'source': 'https://blog.gopenai.com/paper-review-llama-2-open-foundation-and-fine-tuned-chat-models-23e539522acb'}, page_content='Please enable cookies.\n\nSorry, you have been blocked\n\nYou are unable to access medium.com\n\nWhy have I been blocked?\n\nThis website is using a security service to protect itself from online attacks. The action you just performed triggered the security solution. There are several actions that could trigger this block including submitting a certain word or phrase, a SQL command or malformed data.\n\nWhat can I do to resolve this?\n\nYou can email the site owner to let them know you were blocked. Please include what you were doing when this page came up and the Cloudflare Ray ID found at the bottom of this page.\n\nCloudflare Ray ID: a34be5376bb2e66c • Your IP: 34.125.192.214 • Performance & security by Cloudflare')

#Split the Text into Chunks

In [20]:
text_splitter=CharacterTextSplitter(separator='\n',
                                    chunk_size=1000,
                                    chunk_overlap=200)

In [22]:
text_chunks=text_splitter.split_documents(data)
print(type(text_chunks))
len(text_chunks)

<class 'list'>


61

In [23]:
text_chunks[0]

Document(metadata={'source': 'https://blog.gopenai.com/paper-review-llama-2-open-foundation-and-fine-tuned-chat-models-23e539522acb'}, page_content='Please enable cookies.\nSorry, you have been blocked\nYou are unable to access medium.com\nWhy have I been blocked?\nThis website is using a security service to protect itself from online attacks. The action you just performed triggered the security solution. There are several actions that could trigger this block including submitting a certain word or phrase, a SQL command or malformed data.\nWhat can I do to resolve this?\nYou can email the site owner to let them know you were blocked. Please include what you were doing when this page came up and the Cloudflare Ray ID found at the bottom of this page.\nCloudflare Ray ID: a34be5376bb2e66c • Your IP: 34.125.192.214 • Performance & security by Cloudflare')

In [24]:
text_chunks[1]

Document(metadata={'source': 'https://www.mosaicml.com/blog/mpt-7b'}, page_content='Skip to main content\nAI Research\nMay 5, 2023\nIntroducing MPT-7B: A New Standard for Open-Source, Commercially Usable LLMs\nby The Databricks AI Research Team\nIntroducing MPT-7B, the first entry in our MosaicML Foundation Series. MPT-7B is a transformer trained from scratch on 1T tokens of text and code. It is open source, available for commercial use, and matches the quality of LLaMA-7B. MPT-7B was trained on the MosaicML platform in 9.5 days with zero human intervention at a cost of ~$200k.\nLarge language models (LLMs) are changing the world, but for those outside well-resourced industry labs, it can be extremely difficult to train and deploy these models. This has led to a flurry of activity centered on open-source LLMs, such as the LLaMA series from Meta, the Pythia series from EleutherAI, the StableLM series from StabilityAI, and the OpenLLaMA model from Berkeley AI Research.')

#Downlaod the Huuging face Embeddings

In [32]:
embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [33]:
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [35]:
query_result=embeddings.embed_query("Hello World hhhhhhhhhhhhhh")
print(len(query_result))

384


#Convert the Text Chunks into Embeddings and Create a knowledge base